# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display metadata name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields.

In [ ]:
# Inspect available record sets in the dataset
record_sets = dataset.metadata.recordSet

if record_sets:
    print("Record Sets Found:")
    for rs in record_sets:
        print(f"@id: {rs['@id']}, name: {rs.get('name', rs['@id'])}")
        # If fields exist, print them
        fields = rs.get('field', [])
        for f in fields:
            print(f"  Field @id: {f['@id']}, name: {f.get('name', f['@id'])}")
else:
    print("No record sets listed in metadata. Let's try to fetch available records and infer possible record set @ids.")

# Try to get possible record set @ids from the dataset
possible_record_sets = dataset.record_sets()
print("Possible record set @ids from mlcroissant:")
for rid in possible_record_sets:
    print(f"@id: {rid}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We select the main survey record set for extraction. The record set and its fields/columns are referenced by their `@id` fields as per best Croissant practices.


In [ ]:
# Use the main survey record set @id discovered. Assume it is 'https://api.app.sen.science/frontiers/7160411/d4c7e112-8090-4dab-88c9-8e2edee8c6e1'
main_record_set_id = 'https://api.app.sen.science/frontiers/7160411/d4c7e112-8090-4dab-88c9-8e2edee8c6e1'

# If multiple record sets were found, add more @ids to the list
record_sets_to_extract = [main_record_set_id]
dataframes = {}

for record_set_id in record_sets_to_extract:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in record set {record_set_id}:")
        print(df.columns.tolist())
        print(df.head(3))
    else:
        print(f"No records found for record set {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below we:
- Filter respondents based on GAD-7 score (`@id`: 'https://api.app.sen.science/frontiers/7160411/gad7_score')
- Normalize the score
- Group by gender (`@id`: 'https://api.app.sen.science/frontiers/7160411/gender')

In [ ]:
# Begin EDA on the main record set
df = dataframes.get(main_record_set_id)

# Reference fields by their @id
gad7_score_id = 'https://api.app.sen.science/frontiers/7160411/gad7_score'
gender_id = 'https://api.app.sen.science/frontiers/7160411/gender'

if df is not None and gad7_score_id in df.columns:
    # Filter records with GAD-7 score > 10
    threshold = 10
    filtered_df = df[df[gad7_score_id] > threshold]
    print(f"Filtered records with {gad7_score_id} > {threshold}:")
    print(filtered_df[[gad7_score_id]].head())

    # Normalize the GAD-7 score
    filtered_df[f"{gad7_score_id}_normalized"] = (
        filtered_df[gad7_score_id] - filtered_df[gad7_score_id].mean()
    ) / filtered_df[gad7_score_id].std()
    
    print(f"Normalized {gad7_score_id} for filtered records:")
    print(filtered_df[[gad7_score_id, f"{gad7_score_id}_normalized"]].head())

    # Group by gender
    if gender_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(gender_id)[gad7_score_id].mean()
        print(f"Grouped mean GAD-7 score by {gender_id}:")
        print(grouped_df)
else:
    print(f"Field {gad7_score_id} not present in columns for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below we plot the distribution of GAD-7 scores, colored by gender, using the field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize GAD-7 score distribution
if df is not None and gad7_score_id in df.columns and gender_id in df.columns:
    plt.figure(figsize=(8,6))
    sns.histplot(data=df, x=gad7_score_id, hue=gender_id, bins=15, kde=True, palette='Set2')
    plt.title(f"Distribution of GAD-7 Scores by {gender_id}")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Count")
    plt.show()
else:
    print("Cannot visualize due to missing fields in DataFrame.")

## 6. Conclusion
In this notebook, we loaded a FAIR² survey dataset for Kilifi County, Kenya, reviewed record set structure and IDs, extracted data, performed basic filtering and normalization based on GAD-7 scores, grouped results by gender, and visualized distribution patterns. This workflow illustrates how to leverage `mlcroissant` for AI-ready, standards-compliant data exploration.
